# 00 — jnwb discovery and addressing

Demonstrates the **address-first** jnwb workflow on local NWB files.

**Rule:** this notebook imports package APIs only. It defines no functions or classes.

APIs shown:
- `jnwb.list_nwb_files`
- `jnwb.inspect_nwb`
- `jnwb.build_session_manifest`
- `jnwb.address_events`
- `jnwb.address_signals`

Set `OMISSION_NWB_ROOT` to your NWB directory. If unset, cells skip safely when no data are available.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "src").exists() and (REPO.parent / "src").exists():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import jnwb

NWB_ROOT = Path(os.environ.get("OMISSION_NWB_ROOT", "D:/analysis/nwb"))
print(f"Repo: {REPO}")
print(f"NWB root: {NWB_ROOT}")
print(f"jnwb public API: {jnwb.__all__}")

In [ ]:
if not NWB_ROOT.exists():
    print("SKIP: NWB root not found. Set OMISSION_NWB_ROOT to run discovery.")
    files = []
else:
    files = jnwb.list_nwb_files(NWB_ROOT)
    print(f"Found {len(files)} NWB files")
    if files:
        rec = files[0]
        print(f"First session: {rec.session_id} subject={rec.subject}")
        print(f"  has_spk={rec.has_spk} has_lfp={rec.has_lfp} has_muae={rec.has_muae}")

In [ ]:
if NWB_ROOT.exists() and files:
    summary = jnwb.inspect_nwb(files[0])
    print("inspect_nwb keys:", sorted(summary.keys())[:12], "...")
    manifest_path = REPO / "outputs" / "jnwb_notebooks" / "session_manifest.csv"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_df = jnwb.build_session_manifest(files, out=manifest_path)
    print(f"build_session_manifest -> {manifest_path} ({len(manifest_df)} rows)")
else:
    print("SKIP: inspect_nwb / build_session_manifest require NWB files.")

In [ ]:
AFAMILY = ["AAAB", "AXAB", "AAXB", "AAAX"]

if NWB_ROOT.exists() and files:
    ev = jnwb.address_events(
        files[:1],
        conditions=AFAMILY,
        anchor="p1",
        correct=True,
    )
    print(f"address_events sessions={ev.sessions}")
    print(f"  conditions={ev.conditions} anchor={ev.anchor} p1_code={ev.p1_code}")
    if ev.sessions:
        skey = ev.sessions[0]
        n_events = len(ev.events_by_session.get(skey, []))
        print(f"  events in {skey}: {n_events}")
else:
    print("SKIP: address_events requires NWB files.")

In [ ]:
if NWB_ROOT.exists() and files:
    for signal in ("SPK", "LFP", "MUAe"):
        try:
            sig = jnwb.address_signals(files[:1], signal=signal, require_area=False, max_items=4)
            skey = sig.sessions[0]
            print(
                f"address_signals {signal}: n_ids={len(sig.ids_by_session[skey])} "
                f"path={sig.object_paths.get(skey)}"
            )
        except Exception as exc:
            print(f"address_signals {signal}: blocked ({exc})")
else:
    print("SKIP: address_signals requires NWB files.")
